# Rule-Based KIE Baseline

This notebook evaluates the first interpretable Key Information Extraction baseline on validated SROIE receipts. It consumes OCR text, words, bounding boxes, and confidence values from the grayscale+contrast OCR run. No OCR output, raw data, field annotations, or models are modified.

In [5]:
import importlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import src.extraction.normalization as extraction_normalization
import src.extraction.layout as extraction_layout
import src.extraction.rules as extraction_rules
import src.extraction.extractor as extraction_extractor
import src.extraction as extraction
import src.evaluation.kie as kie_evaluation

importlib.reload(extraction_normalization)
importlib.reload(extraction_layout)
importlib.reload(extraction_rules)
importlib.reload(extraction_extractor)
importlib.reload(extraction)
importlib.reload(kie_evaluation)
from src.data.sroie import load_sroie_dataset
from src.evaluation.kie import evaluate_extractions
from src.extraction import RuleBasedExtractor
from src.extraction.normalization import values_match
from src.ocr import OCRResult, OCRWord

DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw'
OCR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sroie_ocr_preprocessed_gray_contrast.jsonl'
FIELDS = ['company', 'date', 'address', 'total']

dataset = load_sroie_dataset(DATASET_ROOT, strict=False)
assert len(dataset) == 624, f'Expected 624 validated samples, found {len(dataset)}'
if not OCR_PATH.is_file():
    raise FileNotFoundError(f'Grayscale+contrast OCR results not found: {OCR_PATH}')
print(f'Validated samples: {len(dataset)}')
print(f'OCR source: {OCR_PATH.relative_to(PROJECT_ROOT)}')

Validated samples: 624
OCR source: data\processed\sroie_ocr_preprocessed_gray_contrast.jsonl


## Load OCR results

The adapter preserves the existing OCR result structure. The extractor receives the original OCR text and spatial word records without rewriting them.

In [6]:
def load_ocr_results(path: Path) -> list[OCRResult]:
    results = []
    with path.open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'OCR line {line_number} is not an object')
            words = tuple(
                OCRWord(
                    text=str(word['text']),
                    bbox=tuple(int(value) for value in word['bbox']),
                    confidence=word.get('confidence'),
                )
                for word in record.get('words', [])
            )
            results.append(
                OCRResult(
                    text=str(record.get('text', '')),
                    words=words,
                    status=record.get('status', 'failure'),
                    error_type=record.get('error_type'),
                    error_message=record.get('error_message'),
                )
            )
    return results

with OCR_PATH.open(encoding='utf-8') as stream:
    ocr_records = [json.loads(line) for line in stream if line.strip()]
ocr_results = load_ocr_results(OCR_PATH)
assert len(ocr_results) == len(dataset) == len(ocr_records)
print(f'OCR results loaded: {len(ocr_results)}')
print(pd.Series([result.status for result in ocr_results]).value_counts().rename_axis('status').to_frame('count'))

OCR results loaded: 624
         count
status        
success    624


## Extract and evaluate

Rules are deterministic and field-specific. Matching is handled by the independent evaluator, including conservative monetary normalization for `total`.

In [7]:
extractor = RuleBasedExtractor()
predictions = [extractor.extract(result) for result in ocr_results]
ground_truth = dataset.to_dict('records')
metrics = evaluate_extractions(predictions, ground_truth)
display(metrics[['exact_match', 'precision', 'recall', 'f1']])
print(f"Macro-average F1: {metrics.loc['macro_avg', 'f1']:.4f}")

,exact_match,precision,recall,f1
field,,,,
company,0.248397,0.280899,0.280449,0.280674
date,0.613782,0.818376,0.613782,0.701465
address,0.054487,0.115894,0.112179,0.114007
total,0.213141,0.516484,0.301282,0.380567
macro_avg,0.282452,0.432913,0.326923,0.369178


Macro-average F1: 0.3692


## Representative examples

Successful and failed examples retain the image ID, ground truth, prediction, and original OCR text for inspection.

In [8]:
def example_frame(indices: list[int]) -> pd.DataFrame:
    rows = []
    for index in indices:
        prediction = predictions[index].fields()
        rows.append({
            'image_id': dataset.iloc[index]['image_id'],
            'ground_truth': {field: dataset.iloc[index][field] for field in FIELDS},
            'prediction': {field: prediction[field].value for field in FIELDS},
            'ocr_text': ocr_results[index].text,
        })
    return pd.DataFrame(rows)


def is_success(index: int) -> bool:
    prediction = predictions[index].fields()
    return all(
        values_match(
            dataset.iloc[index][field],
            prediction[field].value,
            monetary=field == 'total',
        )
        for field in FIELDS
    )


successful_indices = [index for index in range(len(predictions)) if is_success(index)][:3]
failed_indices = [index for index in range(len(predictions)) if not is_success(index)][:5]
print('Successful examples')
display(example_frame(successful_indices))
print('Failed examples')
display(example_frame(failed_indices))

Successful examples


,image_id,ground_truth,prediction,ocr_text
0,X51005663272,"{'company': 'WAHIN HARDWARE SDN BHD', 'date': ...","{'company': 'WAHIN HARDWARE SDN BHD', 'date': ...",3 1 8[r012\n¥\nWAHIN HARDWARE SDN BHD\n148481-...
1,X51005724621,"{'company': 'KEDAI PAPAN YEW CHUAN', 'date': '...","{'company': 'KEDAI PAPAN YEW CHUAN', 'date': '...",RO\nKEDAI PAPAN YEW CHUAN\n(0005583085-K)\nLOT...
2,X51006556830,"{'company': 'PINGHWAI TRADING SDN BHD', 'date'...","{'company': 'PINGHWAI TRADING SDN BHD', 'date'...","PINGHWAI TRADING SDN BHD\n(1211036-W)\nNO.2, J..."


Failed examples


,image_id,ground_truth,prediction,ocr_text
0,X00016469612,"{'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD',...","{'company': 'BOOK TA -K (TAMAN DAYA) SDN BHD',...",tan woon yann\nBOOK TA -K (TAMAN DAYA) SDN BHD...
1,X00016469619,"{'company': 'INDAH GIFT & HOME DECO', 'date': ...","{'company': 'INDAH GIFT & HOME BECO', 'date': ...","tan woon yann\nINDAH GIFT & HOME BECO\n27, JAL..."
2,X00016469620,"{'company': 'MR D.I.Y. (JOHOR) SDN BHD', 'date...","{'company': 'MR O.1.¥. (JOHOR) SDN BHD', 'date...",tan woon yann\nMR O.1.¥. (JOHOR) SDN BHD\n(CO....
3,X00016469622,"{'company': 'YONGFATT ENTERPRISE', 'date': '25...","{'company': 'YONGFATT ENTERPRISE', 'date': '25...",tan woon yann\nYONGFATT ENTERPRISE\n(4M0517726...
4,X00016469623,"{'company': 'MR D.I.Y. (M) SDN BHD', 'date': '...","{'company': '» (4) SON BHD', 'date': '18-11-18...",tan woon yann\n» (4) SON BHD\n86067 1-D)\ndG)-...


## Conclusion

This rule-based baseline is transparent and useful for measuring progress, but receipt layouts and OCR errors make fixed header, date, address, and total rules brittle. The next modeling phase should compare this baseline with NLP and layout-aware approaches while preserving the same `ExtractionResult` and evaluation interface.